In [1]:
from langchain_community.document_loaders import PyPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_community.vectorstores import FAISS 
from langchain.chains import RetrievalQA
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document
import numpy as np


In [2]:
load_dotenv()
loader1 = PyPDFLoader("history_of_ai.pdf")
loader2 = PyPDFLoader("future_of_ai.pdf")

history_docs = loader1.load()
future_docs = loader2.load()

docs = history_docs + future_docs
print(f"Total pages loaded: {len(docs)}")

Total pages loaded: 16


In [3]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50) 
chunks = splitter.split_documents(docs) 
print(f"Total chunks created: {len(chunks)}")
type(chunks[0])

Total chunks created: 156


langchain_core.documents.base.Document

In [4]:
chunks[2].page_content
chunks[2].metadata

{'source': 'history_of_ai.pdf', 'page': 0}

In [5]:

embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=512)
vectorstore = FAISS.from_documents(chunks, embeddings)

In [6]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":4}) 


In [13]:
# Create RAG chain 

llm = ChatOpenAI(model="gpt-4o-mini") 
rag_chain = RetrievalQA.from_chain_type( 
    llm=llm, 
    retriever=retriever 
) 


Submit the query: "Explain the evolution of AI in a few sentences."
Verify that the system:
Retrieves relevant information from both PDFs.
Combines historical and future perspectives of AI.
Produces a concise, coherent answer.

In [15]:

while True:
    question = input("\nAsk a question (or type 'exit'): ")

    if question.lower() == "exit":
        break

    response = rag_chain.invoke({"query": question.lower()})

    print("\n=== ANSWER ===\n")
    print(response)


=== ANSWER ===

{'query': 'explain evolution of ai', 'result': 'The evolution of artificial intelligence (AI) has been marked by several key developments and milestones. Initially, AI research began in the mid-20th century, focusing on symbolic reasoning and problem-solving. Early systems were rule-based and relied heavily on human expertise to encode knowledge.\n\nAs computational power increased, so did the complexity of AI systems. The introduction of machine learning in the 1980s allowed AI to learn from data rather than just following predefined rules. This shift led to the development of algorithms that could improve their performance over time.\n\nIn the 2000s, the advent of big data and advancements in neural networks, particularly deep learning, revolutionized AI. This enabled systems to process vast amounts of unstructured data, leading to breakthroughs in areas like image and speech recognition.\n\nToday, AI technologies are integrated into various applications, such as rec